In [1]:
from pathlib import Path
import atoti as tt
from atoti_jdbc import JdbcLoad

Welcome to Atoti 0.9.9!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


In [2]:
# Tạo session
session = tt.Session.start(
    tt.SessionConfig(
        user_content_storage=Path("./atoti_content")
    )
)

# Kiểm tra URL app Atoti
print(session.url)

http://localhost:49985


In [3]:
'''
import atoti as tt
from pathlib import Path

with tt.Session.start(
    tt.SessionConfig(user_content_storage=Path("./atoti_content"))
) as session:
    cube = session.create_cube(...)
    # code xử lý ở đây
    ...
# Khi thoát khỏi khối with, session.close() được gọi tự động
'''

'\nimport atoti as tt\nfrom pathlib import Path\n\nwith tt.Session.start(\n    tt.SessionConfig(user_content_storage=Path("./atoti_content"))\n) as session:\n    cube = session.create_cube(...)\n    # code xử lý ở đây\n    ...\n# Khi thoát khỏi khối with, session.close() được gọi tự động\n'

In [3]:
# ===============================================
# 1️⃣ Thông tin kết nối PostgreSQL
# ===============================================
POSTGRES = {
    "host": "localhost",
    "port": 5433,
    "db": "cars",
    "user": "admin",
    "password": "admin123",
}

# Chuỗi kết nối JDBC — phải có prefix "jdbc:postgresql://"
postgres_url = (
    f"jdbc:postgresql://{POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['db']}?"
    f"user={POSTGRES['user']}&password={POSTGRES['password']}"
)


In [4]:
from sqlalchemy import create_engine, MetaData

# 1️⃣ Kết nối PostgreSQL
url = (
    f"postgresql+psycopg2://{POSTGRES['user']}:{POSTGRES['password']}"
    f"@{POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['db']}"
)
engine = create_engine(url)

# 2️⃣ Đọc metadata của schema
metadata = MetaData()
metadata.reflect(engine, schema="marts")

# --- Tạo dict bảng với primary key do mình tự gán ---
# Ví dụ: tables_pk = {"fact_sales": "sales_id", "dim_customer": "customer_id"}
tables_pk = {
    "dim_car_details": "car_details_id",
    "dim_car_general": "car_general_id",
    "dim_car_specs": "car_specs_id",
    "dim_date": "date_id",
    "fact_car_listing": "id"
}

# --- Load dữ liệu vào Atoti ---
tables = {}
for table_fullname, table in metadata.tables.items():
    clean_name = table_fullname.split(".")[-1]  # chỉ tên bảng

    # Nếu bảng không có trong dict tables_pk thì bỏ qua
    if clean_name not in tables_pk:
        print(f"⚠️ Skipping table {clean_name}, no primary key assigned")
        continue

    pk = tables_pk[clean_name]

    query = f"SELECT * FROM {table_fullname}"  # vẫn giữ schema trong query
    jdbc_load = JdbcLoad(query=query, url=postgres_url, driver="org.postgresql.Driver")

    data_types = session.tables.infer_data_types(jdbc_load)
    print(data_types)

    # Tạo table trong Atoti
    tables[clean_name] = session.create_table(
        clean_name,
        data_types=data_types,
        keys={pk} if pk else set(),
        default_values={
            # Default values cho scalar types
            **{col_name: 0 for col_name in data_types 
               if data_types[col_name] in ["int", "long"]},
            **{col_name: 0.0 for col_name in data_types 
               if data_types[col_name] in ["float", "double"]},
            # Default values cho array types
            **{col_name: [0] for col_name in data_types 
               if data_types[col_name] in ["int[]", "long[]"]},
            **{col_name: [0.0] for col_name in data_types 
               if data_types[col_name] in ["float[]", "double[]"]},
        }
    )
    tables[clean_name].load(jdbc_load)
    print(f"✅ Loaded table: {clean_name} | PK: {pk}")
    print(tables[clean_name].head(3).sort_index())

{'car_details_id': 'long', 'year': 'long', 'origin': 'String'}
✅ Loaded table: dim_car_details | PK: car_details_id
                year              origin
car_details_id                          
7               1995           nhập khẩu
8               1996  lắp ráp trong nước
56              2021  lắp ráp trong nước
{'car_general_id': 'long', 'brand': 'String', 'model': 'String', 'body_style': 'String'}
✅ Loaded table: dim_car_general | PK: car_general_id
               brand model body_style
car_general_id                       
5               audi    a3  Hatchback
6               audi    a3      Sedan
10              audi    a6      Sedan
{'car_specs_id': 'long', 'transmission': 'String', 'engine': 'String', 'drivetrain': 'String', 'exterior_color': 'String', 'interior_color': 'String'}
✅ Loaded table: dim_car_specs | PK: car_specs_id
             transmission engine drivetrain exterior_color interior_color
car_specs_id                                                             

In [7]:
dim_tables = {
    "dim_car_details": "car_details_id",
    "dim_car_general": "car_general_id",
    "dim_car_specs": "car_specs_id",
    "dim_date": "date_id"
}

for dim_name, key in dim_tables.items():
    dim_table = session.tables[dim_name]
    tables['fact_car_listing'].join(dim_table, tables['fact_car_listing'][key] == dim_table[key])

In [28]:
session.tables.schema

```mermaid
erDiagram
  "dim_car_general" {
    non-null long PK "car_general_id"
    non-null String "brand"
    non-null String "model"
    non-null String "body_style"
  }
  "dim_car_specs" {
    non-null long PK "car_specs_id"
    non-null String "transmission"
    non-null String "engine"
    non-null String "drivetrain"
    non-null String "exterior_color"
    non-null String "interior_color"
  }
  "dim_date" {
    non-null LocalDate PK "date_id"
    non-null long "day_value"
    non-null long "month_value"
    non-null long "year_value"
  }
  "fact_car_listing" {
    non-null long PK "id"
    non-null long "price"
    non-null long "mileage"
    non-null long "car_general_id"
    non-null long "car_details_id"
    non-null long "car_specs_id"
    non-null LocalDate "date_id"
  }
  "dim_car_details" {
    non-null long PK "car_details_id"
    non-null long "year"
    non-null String "origin"
  }
  "fact_car_listing" }o--o| "dim_car_details" : "car_details_id == car_details_id"
  "fact_car_listing" }o--o| "dim_car_general" : "car_general_id == car_general_id"
  "fact_car_listing" }o--o| "dim_car_specs" : "car_specs_id == car_specs_id"
  "fact_car_listing" }o--o| "dim_date" : "date_id == date_id"
```


In [8]:
# ===============================================
# 7️⃣ Tạo cube và mở UI
# ===============================================
cube = session.create_cube(tables["fact_car_listing"], mode='no_measures')

In [9]:
# Aliasing the hierarchies property to a shorter variable name because we will use it a lot.
h = cube.hierarchies
h

{('dim_car_specs', 'engine'): <atoti.hierarchy.Hierarchy object at 0x1216eabf0>, ('dim_car_specs', 'exterior_color'): <atoti.hierarchy.Hierarchy object at 0x1216e91e0>, ('dim_car_specs', 'drivetrain'): <atoti.hierarchy.Hierarchy object at 0x1216e8d30>, ('dim_car_specs', 'transmission'): <atoti.hierarchy.Hierarchy object at 0x1216e95d0>, ('dim_car_specs', 'interior_color'): <atoti.hierarchy.Hierarchy object at 0x1216e8f40>, ('dim_car_details', 'origin'): <atoti.hierarchy.Hierarchy object at 0x1216e8610>, ('dim_car_general', 'body_style'): <atoti.hierarchy.Hierarchy object at 0x1216eb400>, ('dim_car_general', 'brand'): <atoti.hierarchy.Hierarchy object at 0x1216e8730>, ('dim_car_general', 'model'): <atoti.hierarchy.Hierarchy object at 0x1216eb6a0>, ('fact_car_listing', 'date_id'): <atoti.hierarchy.Hierarchy object at 0x1216ebf10>, ('fact_car_listing', 'id'): <atoti.hierarchy.Hierarchy object at 0x1216eaad0>}

In [10]:
h.update(
    {
        ('dim_car_details', 'year'): [tables['dim_car_details']['year']],
        ('dim_date', 'day_value'): [tables['dim_date']['day_value']],
        ('dim_date', 'month_value'): [tables['dim_date']['month_value']],
        ('dim_date', 'year_value'): [tables['dim_date']['year_value']]
    }
)

In [32]:
h

{('dim_car_specs', 'engine'): <atoti.hierarchy.Hierarchy object at 0x1264572b0>, ('dim_car_specs', 'exterior_color'): <atoti.hierarchy.Hierarchy object at 0x1264556f0>, ('dim_car_specs', 'drivetrain'): <atoti.hierarchy.Hierarchy object at 0x126454280>, ('dim_car_specs', 'transmission'): <atoti.hierarchy.Hierarchy object at 0x126457130>, ('dim_car_specs', 'interior_color'): <atoti.hierarchy.Hierarchy object at 0x126455030>, ('dim_date', 'day_value'): <atoti.hierarchy.Hierarchy object at 0x126454a30>, ('dim_date', 'month_value'): <atoti.hierarchy.Hierarchy object at 0x126457f40>, ('dim_date', 'year_value'): <atoti.hierarchy.Hierarchy object at 0x126455f90>, ('dim_car_details', 'origin'): <atoti.hierarchy.Hierarchy object at 0x12645c2e0>, ('dim_car_details', 'year'): <atoti.hierarchy.Hierarchy object at 0x12645ece0>, ('dim_car_general', 'body_style'): <atoti.hierarchy.Hierarchy object at 0x12645fe50>, ('dim_car_general', 'brand'): <atoti.hierarchy.Hierarchy object at 0x12644ae60>, ('dim_car_general', 'model'): <atoti.hierarchy.Hierarchy object at 0x126449570>, ('fact_car_listing', 'date_id'): <atoti.hierarchy.Hierarchy object at 0x126448250>, ('fact_car_listing', 'id'): <atoti.hierarchy.Hierarchy object at 0x12644bf70>}

In [11]:
m = cube.measures
m

{'contributors.COUNT': <atoti.measure.Measure object at 0x12173ab90>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x121739f90>}

In [12]:
m["price.SUM"] = tt.agg.sum(tables['fact_car_listing']["price"])
m["price.MEAN"] = tt.agg.mean(tables['fact_car_listing']["price"])

m["mileage.SUM"] = tt.agg.sum(tables['fact_car_listing']["mileage"])
m["mileage.MEAN"] = tt.agg.mean(tables['fact_car_listing']["mileage"])

In [35]:
m

{'contributors.COUNT': <atoti.measure.Measure object at 0x126514670>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x126516fb0>, 'price.SUM': <atoti.measure.Measure object at 0x1265175e0>, 'price.MEAN': <atoti.measure.Measure object at 0x126514430>, 'mileage.SUM': <atoti.measure.Measure object at 0x126515d20>, 'mileage.MEAN': <atoti.measure.Measure object at 0x126517ca0>}

In [13]:
session.close()